In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
dataset = load_dataset("thangvip/vietnamese-legal-qa")
dataset

DatasetDict({
    train: Dataset({
        features: ['doc_name', 'doc_type_name', 'article_content', 'generated_qa_pairs', 'generation_time'],
        num_rows: 9715
    })
})

In [ ]:
# Work on the train split (or current dataset if already flattened)
train_ds = dataset["train"] if hasattr(dataset, "keys") and "train" in dataset else dataset

# Create doc_id if it does not exist.
# If doc_name exists, assign one doc_id per contiguous doc_name block; otherwise fallback to row index.
if "doc_id" not in train_ds.column_names:
    if "doc_name" in train_ds.column_names:
        doc_ids = []
        current_doc_id = -1
        previous_doc_name = None

        for doc_name in train_ds["doc_name"]:
            if doc_name != previous_doc_name:
                current_doc_id += 1
                previous_doc_name = doc_name
            doc_ids.append(f"doc_{current_doc_id}")
    else:
        doc_ids = [f"doc_{i}" for i in range(train_ds.num_rows)]

    train_ds = train_ds.add_column("doc_id", doc_ids)

# Create article_id if it does not exist.
if "article_id" not in train_ds.column_names:
    article_ids = [f"article_{i}" for i in range(train_ds.num_rows)]
    train_ds = train_ds.add_column("article_id", article_ids)

# Put key IDs first while preserving remaining column order.
front_cols = [col for col in ["doc_id", "article_id"] if col in train_ds.column_names]
ordered_columns = front_cols + [
    col for col in train_ds.column_names if col not in front_cols
]

dataset = train_ds.select_columns(ordered_columns)

# Remove columns that are not needed for training.
cols_to_drop = [
    col for col in ["doc_name", "doc_type_name", "generation_time"]
    if col in dataset.column_names
]
dataset = dataset.remove_columns(cols_to_drop)

dataset

Dataset({
    features: ['doc_id', 'article_id', 'article_content', 'generated_qa_pairs'],
    num_rows: 9715
})

In [14]:
unique_doc_ids = set(dataset["doc_id"])
print(f"Unique doc_id count: {len(unique_doc_ids)}")

Unique doc_id count: 328


In [15]:
dataset[0]

{'doc_id': 'doc_0',
 'article_id': 'article_0',
 'article_content': 'Điều 5. Nguyên tắc hội nhập và hợp tác quốc tế về địa chất, khoáng sản\n\n1. Hội nhập và hợp tác quốc tế trong nghiên cứu, điều tra cơ bản địa chất, điều tra địa chất về khoáng sản, hoạt động khoáng sản, quản lý hoạt động khoáng sản phải đặt trong tổng thể chiến lược phát triển kinh tế - xã hội của đất nước trong từng thời kỳ; chiến lược địa chất, khoáng sản và công nghiệp khai khoáng; tuân thủ Hiến pháp, pháp luật Việt Nam, Hiến chương Liên hợp quốc, điều ước quốc tế mà nước Cộng hòa xã hội chủ nghĩa Việt Nam là thành viên, bảo đảm phù hợp với đường lối và chính sách đối ngoại của Việt Nam; bảo đảm nguyên tắc hợp tác bình đẳng, cùng có lợi trên cơ sở tôn trọng độc lập, chủ quyền và toàn vẹn lãnh thổ, không can thiệp vào công việc nội bộ của nhau.\n\n2. Tranh chấp quốc tế về địa chất, khoáng sản được giải quyết thông qua các biện pháp hòa bình, theo thông lệ quốc tế, pháp luật quốc tế và pháp luật của các bên liên qua

In [16]:
# Clean the generated_qa_pairs column to only keep the question and answer fields
cleaned_generated_qa_pairs = [
    [
        {"question": qa_pair["question"], "answer": qa_pair["answer"]}
        for qa_pair in generated_qa_pairs
    ]
    for generated_qa_pairs in dataset["generated_qa_pairs"]
]

dataset = dataset.remove_columns(["generated_qa_pairs"]).add_column(
    "generated_qa_pairs", cleaned_generated_qa_pairs
)

dataset["generated_qa_pairs"][0]

[{'answer': 'Theo Khoản 1 Điều 5 Luật Địa chất và Khoáng sản, hội nhập và hợp tác quốc tế về địa chất, khoáng sản được thực hiện trong các hoạt động sau: nghiên cứu, điều tra cơ bản địa chất; điều tra địa chất về khoáng sản; hoạt động khoáng sản; và quản lý hoạt động khoáng sản.',
  'question': 'Theo Điều 5 Luật Địa chất và Khoáng sản, hội nhập và hợp tác quốc tế về địa chất, khoáng sản được thực hiện trong những hoạt động cụ thể nào?'},
 {'answer': 'Khi thực hiện hội nhập và hợp tác quốc tế về địa chất, khoáng sản, Việt Nam cần tuân thủ một số nguyên tắc và khuôn khổ pháp lý quan trọng. Cụ thể, Khoản 1 Điều 5 quy định việc này phải đặt trong tổng thể chiến lược phát triển kinh tế - xã hội của đất nước, chiến lược địa chất, khoáng sản và công nghiệp khai khoáng. Về khuôn khổ pháp lý, Việt Nam phải tuân thủ Hiến pháp, pháp luật Việt Nam, Hiến chương Liên hợp quốc, và các điều ước quốc tế mà Cộng hòa xã hội chủ nghĩa Việt Nam là thành viên. Đồng thời, việc hợp tác phải bảo đảm phù hợp vớ

In [ ]:
from pathlib import Path
import json

output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

corpus_jsonl_path = output_dir / "corpus.jsonl"
columns_to_save = ["doc_id", "article_id", "article_content", "generated_qa_pairs"]

missing_columns = [col for col in columns_to_save if col not in dataset.column_names]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Save only the reusable fields as one JSON record per line.
with corpus_jsonl_path.open("w", encoding="utf-8") as f:
    for row in dataset:
        record = {col: row[col] for col in columns_to_save}
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(dataset)} records to {corpus_jsonl_path}")

# Create chunk store

In [1]:
def convert_to_chunk_object(doc_id, article_id, chunk_id, text):
    return {
        "doc_id": doc_id,
        "article_id": article_id,
        "chunk_id": chunk_id,
        "text": text
    }

def get_jsonl_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

In [ ]:
import math
import re
import json

chars_per_token = 3
max_tokens = 128
chunk_store = []
overlap_ratio = 0.15

SENTENCE_SPLIT_PATTERN = re.compile(r"(?<=[\.!?;:])\s+")
PARAGRAPH_SPLIT_PATTERN = re.compile(r"\n{2,}")
MAX_CHARS_PER_CHUNK = max_tokens * chars_per_token


def estimate_tokens(text):
    return math.ceil(len(text) / chars_per_token)


def split_sentences(text):
    parts = SENTENCE_SPLIT_PATTERN.split(text.strip())
    return [p.strip() for p in parts if p.strip()]


def append_chunk(doc_id, article_id, text):
    text = text.strip()
    if not text:
        return
    chunk_store.append(convert_to_chunk_object(doc_id, article_id, len(chunk_store), text))


dataset = get_jsonl_data("data/corpus.jsonl")
overlap_chunk = []

for row in dataset:
    doc_id = row["doc_id"]
    article_id = row["article_id"]
    document = row["article_content"]
    paragraphs = PARAGRAPH_SPLIT_PATTERN.split(document)
    chunk_tokens_limit = max_tokens

    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue

        paragraph_tokens = estimate_tokens(paragraph)
        if paragraph_tokens <= chunk_tokens_limit:
            append_chunk(doc_id, article_id, paragraph)
            continue

        # Paragraph is too long, split into sentence-based chunks with overlap.
        sentences = split_sentences(paragraph)
        temp_sentences = []
        temp_tokens = 0

        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue

            sentence_tokens = estimate_tokens(sentence)

            # Fallback: if one sentence alone exceeds the limit, hard-split by length.
            if sentence_tokens > chunk_tokens_limit:
                if temp_sentences:
                    append_chunk(doc_id, article_id, " ".join(temp_sentences))
                    temp_sentences = []
                    temp_tokens = 0

                for i in range(0, len(sentence), MAX_CHARS_PER_CHUNK):
                    append_chunk(doc_id, article_id, sentence[i : i + MAX_CHARS_PER_CHUNK])
                continue

            if temp_tokens + sentence_tokens <= chunk_tokens_limit:
                temp_sentences.append(sentence)
                temp_tokens += sentence_tokens
            else:
                append_chunk(doc_id, article_id, " ".join(temp_sentences))
                overlap_chunk.append((doc_id, article_id, len(chunk_store), paragraph[:30] + "..."))

                # Keep overlap to preserve context between neighboring chunks.
                overlap_tokens_limit = int(overlap_ratio * chunk_tokens_limit)
                overlap_tokens = 0
                overlap_sentences = []

                for prev_sentence in reversed(temp_sentences):
                    prev_sentence_tokens = estimate_tokens(prev_sentence)
                    if overlap_tokens + prev_sentence_tokens <= overlap_tokens_limit:
                        overlap_sentences.insert(0, prev_sentence)
                        overlap_tokens += prev_sentence_tokens
                    else:
                        break

                temp_sentences = overlap_sentences + [sentence]
                temp_tokens = overlap_tokens + sentence_tokens

        if temp_sentences:
            append_chunk(doc_id, article_id, " ".join(temp_sentences))

print("Length of chunk_store:", len(chunk_store))

Length of chunk_store: 123039


In [3]:
chunk_store[:5]

[{'doc_id': 'doc_0',
  'article_id': 'article_0',
  'chunk_id': 0,
  'text': 'Điều 5. Nguyên tắc hội nhập và hợp tác quốc tế về địa chất, khoáng sản'},
 {'doc_id': 'doc_0',
  'article_id': 'article_0',
  'chunk_id': 1,
  'text': '1. Hội nhập và hợp tác quốc tế trong nghiên cứu, điều tra cơ bản địa chất, điều tra địa chất về khoáng sản, hoạt động khoáng sản, quản lý hoạt động khoáng sản phải đặt trong tổng thể chiến lược phát triển kinh tế - xã hội của đất nước trong từng thời kỳ; chiến lược địa chất, khoáng sản và công nghiệp khai khoáng;'},
 {'doc_id': 'doc_0',
  'article_id': 'article_0',
  'chunk_id': 2,
  'text': 'tuân thủ Hiến pháp, pháp luật Việt Nam, Hiến chương Liên hợp quốc, điều ước quốc tế mà nước Cộng hòa xã hội chủ nghĩa Việt Nam là thành viên, bảo đảm phù hợp với đường lối và chính sách đối ngoại của Việt Nam; bảo đảm nguyên tắc hợp tác bình đẳng, cùng có lợi trên cơ sở tôn trọng độc lập, chủ quyền và toàn vẹn lãnh thổ, không can thiệp vào công việc nội bộ của nhau.'},
 {

In [4]:
print("The number of overlap chunks:", len(overlap_chunk))
overlap_chunk[0: 5]

The number of overlap chunks: 25117


[('doc_0', 'article_0', 2, '1. Hội nhập và hợp tác quốc tế...'),
 ('doc_0', 'article_1', 39, '29. Tài nguyên khoáng sản là l...'),
 ('doc_0', 'article_2', 43, '1. Luật này quy định việc điều...'),
 ('doc_0', 'article_3', 56, 'a) Phù hợp với chiến lược, quy...'),
 ('doc_0', 'article_4', 66, '2. Nhà nước đầu tư và tổ chức ...')]

In [ ]:
from pathlib import Path

output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

chunks_jsonl_path = output_dir / "chunk_store.jsonl"
meta_json_path = output_dir / "chunk_store_meta.json"

# Save each chunk as one JSON record per line.
with chunks_jsonl_path.open("w", encoding="utf-8") as f:
    for chunk in chunk_store:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

metadata = {
    "num_chunks": len(chunk_store),
    "chars_per_token": chars_per_token,
    "max_tokens": max_tokens,
    "format": "jsonl",
    "file": str(chunks_jsonl_path),
}

with meta_json_path.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

{
    "chunks_file": str(chunks_jsonl_path),
    "meta_file": str(meta_json_path),
    "num_chunks": len(chunk_store),
}

{'chunks_file': 'processed\\chunk_store.jsonl',
 'meta_file': 'processed\\chunk_store_meta.json',
 'num_chunks': 123039}